<a href="https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/ml-humanities-notebooks/01_Classificazione_linguaggio_depressivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio extra 1 — Che cosa rivelano le parole? 🧭
### Classificare testi con la regressione logistica

**Obiettivo (20–30 minuti).** Partiamo da post pubblici in inglese, già etichettati come *depressive* (1) o *non-depressive* (0), alleniamo un classificatore **TF–IDF + logistic regression** e osserviamo quali parole contribuiscono alle predizioni. Poi verifichiamo un'ipotesi linguistica: le forme **I, me, my, never, always, nothing** sono più frequenti in una delle due categorie?

**Una distinzione fondamentale:** il modello **classifica post etichettati**, non stabilisce se una persona abbia una depressione. Le etichette di questo dataset non sono diagnosi cliniche verificate; provenienza dei post, argomento e criteri di annotazione possono influenzare molto i risultati. **Non usatelo per valutare persone reali**, compagni o studenti.

⚠️ Il dataset può contenere riferimenti a disagio psicologico o suicidio. Per questo laboratorio **non mostriamo né condividiamo il contenuto dei singoli post**: lavoriamo solo su misure aggregate.

**Fonte e attribuzione:** [FishHua86 — Depression Data Analysis and Classification on Reddit Posts](https://github.com/FishHua86/Depression-Data-Analysis-and-Classification-on-Reddit-Posts), file `data/reddit_depression.csv` (fonte secondaria che riporta dati Reddit, descritti nel README come provenienti da Kaggle). Il notebook richiede Internet solo per la prima lettura del CSV.

▶️ In Google Colab: **Ambiente di esecuzione → Esegui tutte**. Poi tornate sui grafici e sulle domande.

## 1. Importiamo gli strumenti

- **pandas** organizza i dati in tabelle.
- **TfidfVectorizer** trasforma parole in numeri.
- **LogisticRegression** prova a distinguere le due etichette.
- **train_test_split** tiene da parte testi che il modello non vedrà durante l'apprendimento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("Librerie importate! 🌱")

## 2. Carichiamo il dataset

Il CSV contiene una colonna **text** e una colonna **label** (0/1). Eliminiamo righe vuote, duplicati esatti e testi brevissimi, senza stampare messaggi personali. Se la fonte non risponde, il codice segnala l'indirizzo del file da aprire.

In [ ]:
URL_DATI = (
    "https://raw.githubusercontent.com/"
    "FishHua86/Depression-Data-Analysis-and-Classification-on-Reddit-Posts/"
    "main/data/reddit_depression.csv"
)

try:
    dati = pd.read_csv(URL_DATI, on_bad_lines="skip")
except Exception as errore:
    raise RuntimeError(
        "Impossibile scaricare il CSV. Verifica Internet e riprova; "
        f"in alternativa, apri: {URL_DATI}"
    ) from errore

assert {"text", "label"}.issubset(dati.columns), "Colonne attese: text e label."

df = dati[["text", "label"]].dropna().copy()
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() >= 30]
df = df.drop_duplicates(subset="text").reset_index(drop=True)

# Limitiamo l'esempio a 4000 testi per velocizzare la lezione.
if len(df) > 4000:
    df = df.sample(n=4000, random_state=42).reset_index(drop=True)

print(f"Post utilizzabili: {len(df)}")
print("Numero di post per etichetta (NON diagnosi):")
print(df["label"].value_counts().sort_index().rename(index={0: "0: non depressivo", 1: "1: depressivo"}))
assert df["label"].nunique() == 2, "Servono entrambe le classi."

## 3. Alleniamo un classificatore supervisionato

**Prima** dividiamo i testi in training (75%) e test (25%). **Poi** impariamo il vocabolario TF–IDF **solo sui testi di training**: questo evita che il test influenzi la preparazione del modello (*data leakage*).

Nella tokenizzazione **teniamo anche parole di una sola lettera**, come `I`, e **non togliamo le stopwords**. Altrimenti perderemmo proprio molti pronomi interessanti.

In [ ]:
indici_train, indici_test = train_test_split(
    df.index,
    test_size=0.25,
    random_state=42,
    stratify=df["label"]
)

X_train = df.loc[indici_train, "text"]
X_test = df.loc[indici_test, "text"]
y_train = df.loc[indici_train, "label"]
y_test = df.loc[indici_test, "label"]

vettorizzatore = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-Z]+\b",  # include anche 'I'
    min_df=3,                            # parola presente in almeno 3 post di training
    max_features=5000,                   # teniamo al massimo 5000 termini
    stop_words=None                      # preserviamo pronomi, avverbi, negazioni
)

X_train_num = vettorizzatore.fit_transform(X_train)
X_test_num = vettorizzatore.transform(X_test)

modello = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modello.fit(X_train_num, y_train)

predizioni = modello.predict(X_test_num)
print("Accuracy:", round(accuracy_score(y_test, predizioni), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, predizioni), 3))
print("Baseline maggioritaria (solo per confronto):", round(y_test.value_counts(normalize=True).max(), 3))

ConfusionMatrixDisplay.from_predictions(
    y_test, predizioni,
    display_labels=["non depressivo", "depressivo"],
    cmap="Greys", values_format="d"
)
plt.title("Test: previsioni vs etichette dei post")
plt.show()

## 4. Quali parole usa il modello?

Nella regressione logistica ogni parola ha un **coefficiente**. A parità delle altre feature:

- coefficiente **positivo** → la parola aumenta lo score associato alla classe 1;
- coefficiente **negativo** → la parola aumenta relativamente lo score della classe 0.

Attenzione: **predittivo non significa clinicamente significativo, né causale**. Anche argomenti e comunità da cui provengono i post possono produrre forti differenze.

In [ ]:
termini = np.array(vettorizzatore.get_feature_names_out())
pesi = modello.coef_[0]

indici_positivi = np.argsort(pesi)[-12:][::-1]
indici_negativi = np.argsort(pesi)[:12]

tabella = pd.DataFrame({
    "più associati a label=1": termini[indici_positivi],
    "coefficiente (+)": np.round(pesi[indici_positivi], 3),
    "più associati a label=0": termini[indici_negativi],
    "coefficiente (−)": np.round(pesi[indici_negativi], 3),
})
display(tabella)

parole_da_osservare = ["i", "me", "my", "myself", "we", "our", "never", "always", "nothing", "not"]
print("\nCoefficienti delle parole della nostra ipotesi:")
for parola in parole_da_osservare:
    if parola in vettorizzatore.vocabulary_:
        posizione = vettorizzatore.vocabulary_[parola]
        print(f"{parola:>9} : {pesi[posizione]:+.3f}")
    else:
        print(f"{parola:>9} : assente dal vocabolario del modello")

## 5. Mettiamo alla prova l'ipotesi: pronomi e avverbi

Ora non guardiamo più i coefficienti del modello: contiamo direttamente le occorrenze delle parole, **per 100 parole del post**. Questo rende confrontabili anche testi di lunghezza diversa. Le medie si calcolano sui post di ciascuna classe.

**Domanda:** `never` è davvero più frequente nei post con etichetta 1? E `I`? Non è detto: controlliamo.

In [ ]:
MARCATORI = ["i", "me", "my", "myself", "we", "our", "never", "always", "nothing", "not"]

def frequenze_per_100(testo):
    # Trasformiamo il testo in una lista di parole inglesi in minuscolo.
    parole = re.findall(r"[a-z]+(?:'[a-z]+)?", testo.lower())
    frequenze = Counter(parole)
    totale = max(1, len(parole))
    return {parola: 100 * frequenze[parola] / totale for parola in MARCATORI}

misure = pd.DataFrame([frequenze_per_100(t) for t in df["text"]])
misure["label"] = df["label"]

medie = misure.groupby("label")[MARCATORI].mean().T
medie.columns = ["label 0", "label 1"]
medie["differenza (1 − 0)"] = medie["label 1"] - medie["label 0"]
display(medie.sort_values("differenza (1 − 0)", ascending=False).round(3))

scelte = ["i", "my", "we", "never", "always", "nothing"]
medie.loc[scelte, ["label 0", "label 1"]].plot(kind="bar", figsize=(9, 4))
plt.ylabel("Occorrenze medie per 100 parole")
plt.title("Confronto aggregato di pronomi e avverbi")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Basta una manciata di marcatori per classificare?

Alleniamo un secondo modello usando **solo le 10 frequenze** che abbiamo appena calcolato. Non è un modello clinico: serve a verificare se questi indicatori stilistici contengono da soli qualche informazione utile.

In [ ]:
X_marcatori_train = misure.loc[indici_train, MARCATORI]
X_marcatori_test = misure.loc[indici_test, MARCATORI]

modello_marcatori = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
modello_marcatori.fit(X_marcatori_train, y_train)
pred_marcatori = modello_marcatori.predict(X_marcatori_test)

print("Solo 10 marcatori — balanced accuracy:",
      round(balanced_accuracy_score(y_test, pred_marcatori), 3))
print("Intero vocabolario — balanced accuracy:",
      round(balanced_accuracy_score(y_test, predizioni), 3))

## 🎲 Mini-esercizio (5 minuti, a coppie)

1. **Prima dei risultati**: scegliete tre parole che, secondo voi, sarebbero indicative della classe 1. Una deve essere un **pronome** e una un **avverbio**.
2. Guardate la tabella e il grafico: in quali casi avevate ragione? Dove i dati contraddicono la vostra aspettativa?
3. Confrontate le due *balanced accuracy*: quanto perdiamo passando da migliaia di parole a soli dieci marcatori?
4. Pensate ad **almeno due spiegazioni alternative** delle differenze (per esempio argomento, lunghezza, comunità d'origine, criteri di etichettatura).

**Esperimento bonus.** Il vocabolario contiene parole troppo facili come `depression` o `suicide`? Provate a escluderle con `stop_words=["depression", "depressed", "suicide", "suicidal", "therapy", "diagnosis"]` nel `TfidfVectorizer`, poi rieseguite le celle 3–4. Come cambiano i risultati?

**Da ricordare:** la psicologia del linguaggio richiede controlli, contesto e attenzione alle persone. Qui abbiamo soltanto un esperimento di NLP su etichette di testi. 🌿